In [ ]:
import torch
import numpy as np
from datasets import load_dataset, DatasetDict, concatenate_datasets
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2CTCTokenizer, Wav2Vec2ForCTC, Wav2Vec2Processor
from huggingface_hub import login
import librosa
import random


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

login(token="REMOVED_SECRET")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

main_seed=42

model_name = "facebook/mms-1b"
model_language = "mansi"

stt_path = "../data/stt_dataset/"
tts_path = "../data/tts_dataset/"
bonus_path = "../data/bonus_dataset/"

In [ ]:
stt_train_dataset = load_dataset('csv', data_files=stt_path+'train/metadata.csv', delimiter='\t', split='train')
stt_test_dataset = load_dataset('csv', data_files=stt_path+'test/metadata.csv', delimiter='\t', split='train')

tts_train_dataset = load_dataset('csv', data_files=tts_path+'train/metadata.csv', delimiter='\t', split='train')
tts_test_dataset = load_dataset('csv', data_files=tts_path+'test/metadata.csv', delimiter='\t', split='train')

bonus_train_dataset = load_dataset('csv', data_files=bonus_path+'train/metadata.csv', delimiter='\t', split='train')
bonus_test_dataset = load_dataset('csv', data_files=bonus_path+'test/metadata.csv', delimiter='\t', split='train')

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

def augment_audio(y, sr=16000):
    aug_type = random.choice(["stretch", "pitch", "noise"])
    
    if aug_type == "stretch":
        rate = random.uniform(0.9, 1.1)
        y = librosa.effects.time_stretch(y=y, rate=rate)
    elif aug_type == "pitch":
        n_steps = random.randint(-2, 2)
        y = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)
    elif aug_type == "noise":
        noise = np.random.randn(len(y))
        y = y + 0.005 * noise
    
    return y


def prepare_dataset(path, batch, apply_augment=False):
    speech_array, _ = librosa.load(path+'wavs/'+batch['audio_file'], sr=16000)
    
    if apply_augment:
        speech_array = augment_audio(speech_array, sr=16000)
    
    batch["input_values"] = feature_extractor(speech_array, sampling_rate=16000).input_values[0]
    return batch


stt_train_augmented = stt_train_dataset.map(lambda batch: prepare_dataset(path=stt_path + 'train/', batch=batch, apply_augment=True))
stt_train_dataset = stt_train_dataset.map(lambda batch: prepare_dataset(path=stt_path + 'train/', batch=batch, apply_augment=False))
stt_train_dataset = concatenate_datasets([stt_train_dataset, stt_train_augmented])

stt_test_dataset = stt_test_dataset.map(lambda batch: prepare_dataset(path=stt_path + 'test/', batch=batch, apply_augment=False))



tts_train_augmented = tts_train_dataset.map(lambda batch: prepare_dataset(path=tts_path + 'train/', batch=batch, apply_augment=True))
tts_train_dataset = tts_train_dataset.map(lambda batch: prepare_dataset(path=tts_path + 'train/', batch=batch, apply_augment=False))
tts_train_dataset = concatenate_datasets([tts_train_dataset, tts_train_augmented])

tts_test_dataset = tts_test_dataset.map(lambda batch: prepare_dataset(path=tts_path + 'test/', batch=batch, apply_augment=False))


bonus_train_augmented = bonus_train_dataset.map(lambda batch: prepare_dataset(path=bonus_path + 'train/', batch=batch, apply_augment=True))
bonus_train_dataset = bonus_train_dataset.map(lambda batch: prepare_dataset(path=bonus_path + 'train/', batch=batch, apply_augment=False))
bonus_train_dataset = concatenate_datasets([bonus_train_dataset, bonus_train_augmented])

bonus_test_dataset = bonus_test_dataset.map(lambda batch: prepare_dataset(path=bonus_path + 'test/', batch=batch, apply_augment=False))

In [ ]:
def has_features(batch):
    result = []
    for f in batch.get("input_values", []):
        if f is None:
            result.append(False)
        else:
            result.append(np.isfinite(f).all())
    return result

stt_train_dataset = stt_train_dataset.filter(has_features, batched=True, num_proc=2)
stt_test_dataset = stt_test_dataset.filter(has_features, batched=True, num_proc=2)
tts_train_dataset = tts_train_dataset.filter(has_features, batched=True, num_proc=2)
tts_test_dataset = tts_test_dataset.filter(has_features, batched=True, num_proc=2)
bonus_train_dataset = bonus_train_dataset.filter(has_features, batched=True, num_proc=2)
bonus_test_dataset = bonus_test_dataset.filter(has_features, batched=True, num_proc=2)

In [ ]:
import json

with open("vocab.txt", "r", encoding="utf-8") as f:
    vocab = [line.rstrip("\n") for line in f]

vocab_dict = {char: i for i, char in enumerate(vocab)}

if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict[" "]
    del vocab_dict[" "]

vocab_dict["[PAD]"] = len(vocab_dict)
vocab_dict["[UNK]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

tokenizer = Wav2Vec2CTCTokenizer(
    "./vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

In [ ]:
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def tokenize(batch):
    with processor.as_target_processor():
        batch["labels"] = processor(batch["text"]).input_ids

    return batch


stt_train_dataset = stt_train_dataset.map(lambda batch: tokenize(batch=batch), batched=True, num_proc=2)
stt_test_dataset = stt_test_dataset.map(lambda batch: tokenize(batch=batch), batched=True, num_proc=2)

stt_train_dataset.set_format(type='torch', columns=['input_values', 'labels'])
stt_test_dataset.set_format(type='torch', columns=['input_values', 'labels'])



tts_train_dataset = tts_train_dataset.map(lambda batch: tokenize(batch=batch), batched=True, num_proc=2)
tts_test_dataset = tts_test_dataset.map(lambda batch: tokenize(batch=batch), batched=True, num_proc=2)

tts_train_dataset.set_format(type='torch', columns=['input_values', 'labels'])
tts_test_dataset.set_format(type='torch', columns=['input_values', 'labels'])



bonus_train_dataset = bonus_train_dataset.map(lambda batch: tokenize(batch=batch), batched=True, num_proc=2)
bonus_test_dataset = bonus_test_dataset.map(lambda batch: tokenize(batch=batch), batched=True, num_proc=2)

bonus_train_dataset.set_format(type='torch', columns=['input_values', 'labels'])
bonus_test_dataset.set_format(type='torch', columns=['input_values', 'labels'])


def non_empty_labels(batch):
    return [
        len(lbl) > 0
        for lbl in batch.get("labels", [])
    ]

stt_train_dataset = stt_train_dataset.filter(non_empty_labels, batched=True, num_proc=4)
stt_test_dataset  = stt_test_dataset.filter(non_empty_labels, batched=True, num_proc=4)
tts_train_dataset = tts_train_dataset.filter(non_empty_labels, batched=True, num_proc=4)
tts_test_dataset  = tts_test_dataset.filter(non_empty_labels, batched=True, num_proc=4)
bonus_train_dataset = bonus_train_dataset.filter(non_empty_labels, batched=True, num_proc=4)
bonus_test_dataset  = bonus_test_dataset.filter(non_empty_labels, batched=True, num_proc=4)


dataset = DatasetDict({
    "train": concatenate_datasets([stt_train_dataset, tts_train_dataset, bonus_train_dataset]),
    "test": concatenate_datasets([stt_test_dataset, tts_test_dataset, bonus_test_dataset]),
})

dataset

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
import evaluate
import re
import unicodedata

def normalize(text: str) -> str:
    text = text.lower()
    
    normalized_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat == "Zs":
            normalized_chars.append(ch)
        elif unicodedata.combining(ch) and ch in "̄":
            normalized_chars.append(ch)
        elif ch in ["ӈ"]:
            normalized_chars.append(ch)

    return re.sub(r"\s+", " ", "".join(normalized_chars)).strip()

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)


    orh_wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    orh_cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)

    pred_str_norm = [normalize(pred) for pred in pred_str]
    label_str_norm = [normalize(label) for label in label_str]

    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [ 
        label_str_norm[i] for i in range(len(label_str_norm))if len(label_str_norm[i]) > 0
    ]

    random_idx = np.random.randint(0, len(pred_ids))

    print("Label IDs:", label_ids[random_idx])
    print("Predicted IDs:", pred_ids[random_idx])
    print("Label text:", label_str[random_idx])
    print("Predicted text:", pred_str[random_idx])
    print("Norm Label text:", label_str_norm[random_idx])
    print("Norm Predicted text:", pred_str_norm[random_idx])

    wer = 100 * metric_wer.compute(predictions=pred_str_norm, references=label_str_norm)
    cer = 100 * metric_cer.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer": wer, "cer": cer, "orh_wer": orh_wer, "orh_cer": orh_cer}

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    model_name ,
    ctc_loss_reduction="mean", 
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size = processor.tokenizer.vocab_size
)
model.config.ctc_zero_infinity = True

In [ ]:
import torch

def is_ctc_ok(example):
    n = len(example["input_values"])
    if n <= 0:
        return False

    feat_len = model._get_feat_extract_output_lengths(
        torch.tensor([n])
    ).item()

    if feat_len <= 0:
        return False

    if feat_len < len(example["labels"]):
        return False

    return True


dataset["train"] = dataset["train"].filter(is_ctc_ok)
dataset["test"]  = dataset["test"].filter(is_ctc_ok)

In [ ]:
from transformers import TrainingArguments, Trainer

model.freeze_feature_extractor()

training_args = TrainingArguments(
    output_dir="./mms_1b_mansi_v1",
    overwrite_output_dir=True,

    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim='adamw_torch',

    group_by_length=True,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    fp16_full_eval=torch.cuda.is_available(),
    gradient_accumulation_steps=4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,

    num_train_epochs=10,
    eval_strategy="steps",
    eval_steps=5000,
    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,
    
    seed=main_seed,
    data_seed=main_seed,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=False,
    hub_private_repo=False,
    hub_always_push=False
)

trainer = Trainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

trainer.save_model("./mms_1b_mansi_v1")
processor.save_pretrained("./mms_1b_mansi_v1")